# Week 10 studio — REFERENCE SOLUTION (instructor-only)

**Task brief:** [`README.md`](README.md) · **Lesson plan:** [`../../weeks/week-10.md`](../../weeks/week-10.md) · **Target app:** [`../../labs/vuln-llm/README.md`](../../labs/vuln-llm/README.md)

Teaching walkthrough of week 10 — *contain what you cannot prevent*. It **imports
the two exploits and the least-privilege policy from [`solution.py`](solution.py)**
— never re-pasting them — so what runs here is exactly what `test_studio.py`
grades. Correctness is verified separately:
`python3 studios/_verify_solutions.py week-10`.

**Laptop-only.** Importing `solution` brings up the same `vuln-llm` app
**in-process** on `127.0.0.1` and forces its **deterministic MOCK backend**
(`OLLAMA_URL` → dead port). No Docker, no Ollama, no GPU.

> ⚠️ **The mock is a FAKE.** It reproduces the RAG/agent injection *logic*
> deterministically — the model is "fooled" every time by construction — so you
> demonstrate the containment **guarantee**, not axis-9 non-determinism. On a real
> Ollama backend the agent plans tools probabilistically; you would run each
> attack ≥5 trials and report reliability. Re-run against a live model to see it.

> Do not distribute. Excluded from students via `studios/.gitignore`.

In [ ]:
# --- bootstrap: week folder (for solution) + repo root (for seclab) ---
import sys, pathlib
here = pathlib.Path.cwd()
week = here if (here / "solution.py").exists() else here / "studios" / "week-10"
root = week.parent.parent
for p in (str(week), str(root)):
    if p not in sys.path:
        sys.path.insert(0, p)

import inspect
import solution                       # importing it boots the in-process mock target
from seclab.attack import Payload, run_payloads, contains_oracle
print("vuln-llm mock up at", solution.BASE, " backend =", solution.get("/health")["backend"])
ALL_TOOLS = solution.get("/policy")["all_tools"]
DANGEROUS = {"lookup_balance", "send_email"}
print("tools:", ALL_TOOLS, "  dangerous:", DANGEROUS)

## Task 1 — RAG exfil: retrieval with no access control is Broken Access Control

`/rag` retrieves the top-k documents by keyword overlap and stuffs them into the
model's context — with **no notion of who is asking**. The `private-vip` note that
should require authorization surfaces for anyone whose question hits its content
words. `rag_exfil_question` targets the *master reconciliation code* the note
alone mentions; the canary oracle confirms the **actual** leak, not a reflection.

In [ ]:
print(inspect.getsource(solution.rag_exfil_question))
q = solution.rag_exfil_question()
r = solution.post("/rag", {"question": q})
print("retrieved:", r["retrieved"])
# mirrors test_rag_exfil_leaks_private_doc
assert "private-vip" in r["retrieved"], "the unscoped retriever must surface private-vip"
findings = run_payloads(
    [Payload(q, intent="exfil private doc", family="rag-access-control")],
    send=lambda question: solution.post("/rag", {"question": question})["answer"],
    oracle=contains_oracle(solution.CANARY))
assert findings[0].confirmed, "canary must actually appear in the answer"
print("RAG exfil CONFIRMED — private-vip surfaced and its canary leaked into the answer")

## Task 2 — Confused deputy: the agent's authority becomes the attacker's

`/agent` has three tools: `get_faq` (harmless), `lookup_balance` (returns the
secret), `send_email` (exfil-capable). The **model decides** which to call; the
**dispatcher enforces** the policy. With the default all-tools policy,
`confused_deputy_request` talks the agent into chaining `lookup_balance` →
`send_email`, borrowing authority you don't have — OWASP **LLM06 Excessive
Agency**. The oracle is the real effect: an email *sent* carrying the canary.

In [ ]:
print(inspect.getsource(solution.confused_deputy_request))
solution.post("/policy", {"allow": ALL_TOOLS})          # default: wide open
req = solution.confused_deputy_request()
r = solution.post("/agent", {"request": req})
executed = [t["tool"] for t in r["transcript"] if t["status"] == "EXECUTED"]
print("executed:", executed, " sent:", r["sent"])
# mirrors test_confused_deputy_exfiltrates_without_policy
assert "send_email" in executed and r["sent"]
assert solution.CANARY in r["sent"][0]["body"], "the email body must carry the canary"
print("confused deputy CONFIRMED — secret exfiltrated to", r["sent"][0]["to"])

## Task 3 — Contain it and prove it: 0% coverage, 100% containment (the guarantee)

`least_privilege_allowlist` returns the **minimum** an FAQ agent needs —
`["get_faq"]`. It is enforced in the **dispatcher (Python), not a prompt**, so it
is a guarantee you can *state*. Watch both halves live: WITHOUT the policy the
attack lands; WITH it the SAME injection is **still planned** by the fooled model
(coverage 0%) yet **denied** by the dispatcher and nothing is sent (containment
100%). A guarantee you can watch **hold while the attack keeps firing**.

In [ ]:
print(inspect.getsource(solution.least_privilege_allowlist))

# Half 1 — WITHOUT the policy, the attack succeeds (the consequence lands).
solution.post("/policy", {"allow": ALL_TOOLS})
before = solution.post("/agent", {"request": req})
assert before["sent"] and solution.CANARY in before["sent"][0]["body"]
print("wide open  -> sent:", before["sent"][0]["to"], "(exfil succeeds)")

# Apply the least-privilege allow-list.
allow = solution.least_privilege_allowlist()
assert DANGEROUS.isdisjoint(allow) and allow            # excludes lookup_balance + send_email
solution.post("/policy", {"allow": allow})

# Half 2 — SAME injection, re-run through the narrowed dispatcher.
after = solution.post("/agent", {"request": req})
planned = [t["tool"] for t in after["transcript"]]
denied  = [t["tool"] for t in after["transcript"] if t["status"] == "DENIED_BY_POLICY"]
# mirrors test_least_privilege_contains_what_it_cannot_prevent
assert DANGEROUS & set(planned), "coverage 0%: the model must STILL plan the forbidden calls"
assert DANGEROUS & set(denied),  "the dispatcher must DENY them"
assert after["sent"] == [],      "containment 100%: nothing leaves the building"
print("least priv -> allow:", allow, " planned:", planned)
print("           -> denied:", denied, " sent:", after["sent"])
print("\n*** GUARANTEE HOLDS: injection still planned (coverage 0%) yet DENIED and nothing",
      "exfiltrated (containment 100%). Enforced in code the model cannot talk past. ***")

## The lesson — GUARANTEE vs RAISES-COST, one level up from week 9

| Control | Prevents the injection? | Contains the damage? | Guarantee you can state? | Bucket |
|---|---|---|---|---|
| Week 9 output filter | no | partially (one surface form) | **no** — bypass by encoding | **RAISES COST** |
| **Week 10 least privilege** | **no** | **yes, fully** | **yes** — enforced in the dispatcher | **GUARANTEE** |

Least privilege has **0% coverage of the injection** (the model is fooled every
time) and **100% containment of the consequence** (nothing leaves). That is not a
weaker result than a filter — it is a *different, stronger kind*, because it is a
guarantee enforced in **code the model cannot talk past**, not a prompt. The
residual risk (a tool whose job *is* to send email) is handled by
human-in-the-loop — the model proposes, a sound check approves.

In [ ]:
print("LESSON: you cannot PREVENT the injection (0% coverage — the model is fooled every time),",
      "but least privilege CONTAINS it (100% — nothing exfiltrated). Unlike week 9's filter this",
      "is a GUARANTEE you can state, because it is enforced OUTSIDE the model, in the dispatcher.")